# 1. Getting started with GEMs and FBA

## Authors
* Sara Benito Vaquerizo, Genome Biology Unit, European Molecular Biology Laboratory (EMBL), Heidelberg
* Francisco Zorrilla, MRC Toxicology Unit, University of Cambridge
* Arianna Basile, MRC Toxicology Unit, University of Cambridge

## Learning outcomes

In this tutorial you will use [cobrapy](https://cobrapy.readthedocs.io/en/latest/) to learn the following:

* **1.1**: Read the models of selected bacterial strains and algae
* **1.2**: Inspect the model - metabolites, reactions and genes
* **1.3**: Run Flux Balance Analysis (FBA) to simulate growth
* **1.4**: Modify medium conditions in your model and assess the differences in growth and production

## Setup

In [1]:
# Import required packages
import cobra
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import (
    single_gene_deletion, 
    single_reaction_deletion, 
    double_gene_deletion,
    double_reaction_deletion
)

## 1.1 Read the models of selected bacterial strains and algae

The [Systems Biology Markup Language](https://sbml.org/) is an XML-based standard format for distributing models which has support for COBRA models through the FBC extension version 3.

Cobrapy has native support for reading and writing SBML with FBCv3. Please note that all id’s in the model must conform to the SBML SID requirements in order to generate a valid SBML file.

All the bacterial models were previously created using CarveMe (https://github.com/cdanielmachado/carveme) and the algae model using CarveMe and PhotoEukStein (https://www.biorxiv.org/content/10.1101/2023.05.22.541783v1) 

In [2]:
## Read model
#model_algae=read_sbml_model("/scratch/sbenito/SBV001_Courses/EMBO_MCD2026/Practical_Metabolic_modelling/Data/Chaetoceros_gracilis.xml")
model_halop=read_sbml_model("../Data/Halopseudomonas.xml")
#model_limno=read_sbml_model("/scratch/sbenito/SBV001_Courses/EMBO_MCD2026/Practical_Metabolic_modelling/Data/Limnobacter_thiooxidans.xml")
#model_yoonia=read_sbml_model("/scratch/sbenito/SBV001_Courses/EMBO_MCD2026/Practical_Metabolic_modelling/Data/Yoonia_vestfoldensis.xml")

## 1.2 Inspect the models - metabolites, reactions, genes and compartments

The reactions, metabolites, genes, and compartments attributes of the cobrapy model are a special type of `list` called a `cobra.DictList`, and each one is made up of `cobra.Reaction`, `cobra.Metabolite`, `cobra.Gene`, `cobra.Compartment` objects, respectively.

In [4]:
#Inpect the model composition
print("Reactions: ",len(model_halop.reactions)) #Total number of reactions
print("Metabolites: ",len(model_halop.metabolites))
print("Genes: ", len(model_halop.genes))
print("Compartments: ",len(model_halop.compartments))

Reactions:  2271
Metabolites:  1521
Genes:  1050
Compartments:  3


When using Jupyter notebooks, this type of information is rendered as a table.

In [5]:
model_halop

Name,GCA_900114765_1_IMG_taxon_2663763599_annotated_assembly_genomic
Memory address,7e2308591400
Number of metabolites,1521
Number of reactions,2271
Number of genes,1050
Number of groups,0
Objective expression,1.0*Growth - 1.0*Growth_reverse_699ae
Compartments,"cytosol, periplasm, extracellular space"


Just like a regular list, objects in the `DictList` can be retrieved by index. For example, to get the 3rd reaction in the model (we use an index value of 2 because of python's 0-indexing):

In [6]:
model_halop.reactions[50]

Reaction identifier,3AMACHYD
Name,3-aminoacrylate hydrolase
Memory address,0x7e23068e6510
Stoichiometry,3amac_c + h2o_c + h_c --> msa_c + nh4_c 3-Aminoacrylate + H2O H2O + H+ --> Malonate semialdehyde + Ammonium
GPR,FOUD01000028_1_20
Lower bound,0.0
Upper bound,1000.0


We can also check a list of reactions in the model

We will consider the reaction glucose 6-phosphate isomerase, which interconverts glucose 6-phosphate and fructose 6-phosphate. The reaction id for this reaction in our test model is PGI. However, if you want to see the IDs of the first `N` number of reactions in the reconstruction, you can run the code below:

In [8]:
reaction_ids = [reaction.id for reaction in model_halop.reactions]
N = 20
reaction_ids[:N]

['12DGR120tipp',
 '12DGR140tipp',
 '12DGR141tipp',
 '12DGR160tipp',
 '12DGR161tipp',
 '12DGR180tipp',
 '12DGR181tipp',
 '12PPDRtex',
 '12PPDRtpp',
 '13PPDH',
 '1P2CBXLCYCL',
 '1P2CBXLR',
 '1PPDCRc',
 '23CTI1',
 '23CTI2',
 '23CTI3',
 '24DECOAR',
 '25DKGLCNt2rpp',
 '25DKGLCNtex',
 '2AACLPGT161']

We can also inspect one specific reaction to see the thermodynamics (bounds: non-reversible, reversible rection), the reactants, products or genes (GPR; Gene 
protein reaction rule). 
For example, let's inspect the reaction Glucose-6-phosphate isomerase 'PGI'

In [9]:
pgi = model_halop.reactions.get_by_id("PGI")
pgi

Reaction identifier,PGI
Name,Glucose-6-phosphate isomerase
Memory address,0x7e23059ca8d0
Stoichiometry,g6p_c <=> f6p_c D-Glucose 6-phosphate <=> D-Fructose 6-phosphate
GPR,FOUD01000020_1_13
Lower bound,-1000.0
Upper bound,1000.0


We can view the full name, the reaction catalyzed, the bounds and the genes as strings.

In [10]:
print(pgi.name)
print(pgi.reaction)
print(pgi.bounds)
print(pgi.genes) 

Glucose-6-phosphate isomerase
g6p_c <=> f6p_c
(-1000.0, 1000.0)
frozenset({<Gene FOUD01000020_1_13 at 0x7e230682b9b0>})


The thermodynamics of a reaction is defined by the upper and lower bounds as large numbers, typically around 1000 are used as infinite limits (unconstrained fluxes). 
Because the `pgi.lower_bound` < 0, and `pgi.upper_bound` > 0, pgi is reversible. Flux can go in one or the other direction. 

In [11]:
print(pgi.lower_bound, "< pgi <", pgi.upper_bound)
print(pgi.reversibility)

-1000.0 < pgi < 1000.0
True


The lower and upper bound of reactions can also be modified, and the reversibility attribute will automatically be updated. The preferred method for manipulating bounds is using reaction.bounds, e.g.

In [12]:
# Save original bounds
old_bounds = pgi.bounds

# Define and print new bounds
pgi.bounds = (0, 1000.0) #This indicates a reaction is non-reversible and goes in one specific direction 
print("New bounds: ",pgi.lower_bound, "< pgi <", pgi.upper_bound)
print("Reversibility after modification:", pgi.reversibility)

# Reset bounds and show reversibility
pgi.bounds = old_bounds
print("Reversibility after resetting:", pgi.reversibility)

New bounds:  0 < pgi < 1000.0
Reversibility after modification: False
Reversibility after resetting: True


## 1.3 Run Flux Balance Analysis (FBA) to simulate growth

In the code chunks below we are running FBA as the mathematical framework to simulate growth and compute the metabolic fluxes. First, we need to set an objective function, typically, the biomass reaction. The function `optimize()` will find the solution that maximixes the flux through the biomass synthesis reaction (growth rate) under the specified constraints (medium, reaction bounds, etc). 

In [13]:
# Run FBA
print(model_halop.objective)  #Print the reaction set as objective function by default - normally the biomass reaction
model_halop.optimize().objective_value #We maximize the objective function reaction - growth rate (1/h)

# Summary methods help visualizing the flux through the extracellular reactions to see what metabolites are being consumed and produced in the model when we maximize for growth
model_halop.summary()

Maximize
1.0*Growth - 1.0*Growth_reverse_699ae


Metabolite,Reaction,Flux,C-Number,C-Flux
LalaDgluMdap_e,EX_LalaDgluMdap_e,5.388,15,0.57%
abg4_e,EX_abg4_e,0.03605,12,0.00%
ade_e,EX_ade_e,11.01,5,0.39%
arg__L_e,EX_arg__L_e,15.94,6,0.68%
ca2_e,EX_ca2_e,0.2805,0,0.00%
chor_e,EX_chor_e,17.43,10,1.24%
cl_e,EX_cl_e,0.2805,0,0.00%
cobalt2_e,EX_cobalt2_e,0.005388,0,0.00%
cu2_e,EX_cu2_e,0.0382,0,0.00%
fe2_e,EX_fe2_e,0.3618,0,0.00%


We can also inspect all reactions that consume or produce a given metabolite in our FBA solution. For example, let't have a look at ATP:

In [14]:
model_halop.metabolites.atp_c.summary()

Percent,Flux,Reaction,Definition
21.08%,1000,ATPS4rpp,adp_c + 4.0 h_p + pi_c <=> atp_c + h2o_c + 3.0 h_c
18.94%,898.6,GALKr,atp_c + gal_c <=> adp_c + gal1p_c + h_c
7.74%,367.4,NDPK5,atp_c + dgdp_c <=> adp_c + dgtp_c
21.08%,1000,PGK,3pg_c + atp_c <=> 13dpg_c + adp_c
21.08%,1000,PYK,adp_c + h_c + pep_c --> atp_c + pyr_c
10.05%,476.9,SUCOAS,atp_c + coa_c + succ_c <=> adp_c + pi_c + succoa_c
0.03%,1.41,URIDK2r,atp_c + dump_c <=> adp_c + dudp_c
Percent,Flux,Reaction,Definition
0.11%,-5.388,3PEPTabcpp,LalaDgluMdap_p + atp_c + h2o_c --> LalaDgluMdap_c + adp_c + h_c + pi_c
18.94%,-898.6,A6PAG,atp_c + gal_c --> adp_c + dgal6p_c + h_c


To inspect the flux of a specific reaction of interest:

In [15]:
model_halop.reactions.PGK.summary()

To inspect fluxes through the first N reactions:

In [16]:
solution = model_halop.optimize()
solution.fluxes[:N]

12DGR120tipp     0.0
12DGR140tipp     0.0
12DGR141tipp     0.0
12DGR160tipp     0.0
12DGR161tipp     0.0
12DGR180tipp     0.0
12DGR181tipp     0.0
12PPDRtex        0.0
12PPDRtpp        0.0
13PPDH           0.0
1P2CBXLCYCL      0.0
1P2CBXLR         0.0
1PPDCRc          0.0
23CTI1           0.0
23CTI2           0.0
23CTI3           0.0
24DECOAR         0.0
25DKGLCNt2rpp    0.0
25DKGLCNtex      0.0
2AACLPGT161      0.0
Name: fluxes, dtype: float64

## 1.4 Modify medium conditions in your model and assess the differences in growth and production

When we used above the summary methods to inspect what are the metabolites being consumed or produced, 'model_halop.summary()', we can observe a very high flux of glucose and other metabolites in the medium. Let's decrease the uptake of glucose in the model and inspect the fluxes again

In [17]:
model_halop.reactions.EX_glc__D_e.bounds = (-20, -10)

print(model_halop.optimize())
print(model_halop.summary())

<Solution 38.797 at 0x7e23051c15b0>
Objective
1.0 Growth = 38.796693605671955

Uptake
------
    Metabolite          Reaction    Flux  C-Number C-Flux
LalaDgluMdap_e EX_LalaDgluMdap_e    3.88        15  0.79%
        abg4_e         EX_abg4_e 0.02595        12  0.00%
         ade_e          EX_ade_e   7.926         5  0.54%
      arg__L_e       EX_arg__L_e   11.48         6  0.94%
         ca2_e          EX_ca2_e  0.2019         0  0.00%
        chor_e         EX_chor_e   12.55        10  1.71%
          cl_e           EX_cl_e  0.2019         0  0.00%
     cobalt2_e      EX_cobalt2_e 0.00388         0  0.00%
         cu2_e          EX_cu2_e 0.02751         0  0.00%
         fe2_e          EX_fe2_e  0.2605         0  0.00%
         fe3_e          EX_fe3_e  0.3029         0  0.00%
         fru_e          EX_fru_e     500         6 40.94%
         gam_e          EX_gam_e   7.759         6  0.64%
      glc__D_e       EX_glc__D_e      20         6  1.64%
       glcur_e        EX_glcur_e   45

We can see that still some of the substrates uptake a very high flux. You can play around modifying the uptake rate of specific reactions
by constraining the bounds of such reaction(s) and see the change in the growth rate

In [18]:
#Let's decrease the maximum uptake and production rate of all exchange reactions ('EX_metabolite_e') to constraint the solution space
for reaction in model_halop.reactions:
    if 'EX_' in reaction.id:
        reaction.bounds=(-10, 10) #Allow a maximum of 10 mmol gDW-1 h-1 of uptake or secretion
print(model_halop.optimize()) #Maximize growth rate under the new constraints
print(model_halop.summary()) #inspect the new uptake and secretion rates of extracellular metabolites


<Solution 2.014 at 0x7e23051bd6a0>
Objective
1.0 Growth = 2.0143066838633854

Uptake
------
    Metabolite          Reaction      Flux  C-Number C-Flux
       5mcsn_e        EX_5mcsn_e        10         5  4.23%
LalaDgluMdap_e EX_LalaDgluMdap_e    0.2014        15  0.26%
        abg4_e         EX_abg4_e  0.001348        12  0.00%
         ade_e          EX_ade_e    0.4115         5  0.17%
      arg__L_e       EX_arg__L_e    0.5958         6  0.30%
      asn__L_e       EX_asn__L_e        10         4  3.39%
          bz_e           EX_bz_e 0.0002014         7  0.00%
         ca2_e          EX_ca2_e   0.01048         0  0.00%
        chol_e         EX_chol_e     5.021         5  2.13%
        chor_e         EX_chor_e    0.2782        10  0.24%
          cl_e           EX_cl_e   0.01048         0  0.00%
     cobalt2_e      EX_cobalt2_e 0.0002014         0  0.00%
         csn_e          EX_csn_e     2.886         4  0.98%
         cu2_e          EX_cu2_e  0.001428         0  0.00%
        

Imagine we have experimental evidence of the metabolites consumed and produced by Halopseudomonas. We know that our species does not produce methanol and neither consumes h2, so we can simulate that by knocking out those specific exchange reactions

In [19]:
model_halop.reactions.EX_meoh_e.knock_out()
model_halop.reactions.EX_h2_e.knock_out()
print(model_halop.optimize())
print(model_halop.summary())

<Solution 1.946 at 0x7e23051f9b50>
Objective
1.0 Growth = 1.9464087057557218

Uptake
------
    Metabolite          Reaction      Flux  C-Number C-Flux
       5mcsn_e        EX_5mcsn_e     2.788         5  1.22%
LalaDgluMdap_e EX_LalaDgluMdap_e    0.1946        15  0.26%
        abg4_e         EX_abg4_e  0.001302        12  0.00%
         ade_e          EX_ade_e    0.3976         5  0.17%
      arg__L_e       EX_arg__L_e    0.5757         6  0.30%
      asn__L_e       EX_asn__L_e        10         4  3.51%
          bz_e           EX_bz_e 0.0001946         7  0.00%
         ca2_e          EX_ca2_e   0.01013         0  0.00%
        chol_e         EX_chol_e     5.413         5  2.38%
        chor_e         EX_chor_e    0.2688        10  0.24%
          cl_e           EX_cl_e   0.01013         0  0.00%
     cobalt2_e      EX_cobalt2_e 0.0001946         0  0.00%
         csn_e          EX_csn_e        10         4  3.51%
         cu2_e          EX_cu2_e   0.00138         0  0.00%
        

The same can occur with genes, we can knock out specific genes, and those will force the knock out of the reactions catalyzed by those genes

In [20]:
print(model_halop.reactions.PGI.genes) #Check the gene of PGI reaction
print(model_halop.genes.FOUD01000020_1_13.reactions) #Reactions in which this gene is involved
model_halop.genes.FOUD01000020_1_13.knock_out() #Knock out that gene
print(model_halop.optimize()) #Check again if growth is affected

frozenset({<Gene FOUD01000020_1_13 at 0x7e230682b9b0>})
frozenset({<Reaction G6PI at 0x7e23060b2f90>, <Reaction G6PI3 at 0x7e23060b3050>, <Reaction PGI at 0x7e23059ca8d0>})
<Solution 1.946 at 0x7e23051e2120>
